# 12.5 一小段聲音有哪些頻率？


聽到一段單音，我們想知道它主要每秒振動幾次；真實聲音還可能同時含多種頻率。可以在每個短框裡問「各種快慢的正弦起伏各佔多少」，再把結果沿時間排列。這叫短時傅立葉變換STFT：每框做頻率分析，保留何時出現哪些頻率。

前置是[12.2取樣率](https://birdhackor.github.io/tiny-perceptron-vlm/12.2.html)、[12.3時間框](https://birdhackor.github.io/tiny-perceptron-vlm/12.3.html)與[12.4加窗](https://birdhackor.github.io/tiny-perceptron-vlm/12.4.html)。傅立葉變換輸出複數，可以先把它看成一起保存的一對數值：實部a與虛部b。Python用j標示虛部，例如 `3+4j`保存3與4。複數的絕對值不是只把負號去掉，而是這一對數值的幅度，算作a平方加b平方後開根號；3平方加4平方是25，開根號得到5。因此這個例子的 `.abs()`得5，再 `.square()`得功率25。這對數值也保存相位，也就是起伏從週期的哪個位置開始；本節只取幅度平方，不討論如何由相位重建波形。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.multimodal import tone

wave = tone(440.0, seconds=0.2)
spectrum = torch.stft(wave, 400, 160, window=torch.hann_window(400), return_complex=True)
power = spectrum.abs().square()
peak = power.mean(-1).argmax().item()
print("頻率格、時間框", tuple(power.shape))
print("最高格編號", peak, "對應Hz", peak * 16000 / 400)

輸入0.2秒、3200點、440Hz單音。STFT的400是每框分析點數n_fft，160是hop，窗用Hann。預設center=True在邊界補資料，讓框中心從開頭開始，所以輸出`(201,21)`，不是直接unfold的框數。201來自400/2+1：實數波形只需保留0到奈奎斯特的非負頻率部分。

第一軸每一格差16000/400=40Hz，格0為0Hz、格11為440Hz、格200為8000Hz。`mean(-1)`沿時間框平均功率，留下各頻率一項，再`argmax`找最大格，輸出11與440.0。這裡的最高格表示主要頻率；附近仍可能有能量，有限框與窗形狀會讓峰不只一個點。

畫時頻圖時，橫軸是時間框、縱軸是頻率格，顏色或亮度才是功率。

上圖畫波形的前0.008秒，縱向是振幅，正負起伏不代表高低音類別；下圖橫軸換成每0.01秒一框的中心時間，縱軸列頻率，440Hz帶用橘色示意功率較強。下圖只畫前五框，中心時間從0到0.04秒；程式分析的完整聲音仍長0.2秒，共21框。下圖亮度是示意，不是這段程式實測矩陣；它幫我們分清兩種圖的軸，而不是增加一份量測。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/multimodal_audio_axes.svg")))

不能把201行當201秒，也不能把21列當21Hz。窗長加大通常能讓頻率格更密，但每框覆蓋更久，瞬間變化可能更難定位，這是一種時間與頻率觀察尺度的取捨。

若聲音是450Hz而不是440，400點分析的格子仍只有440、480等40Hz間距，不會自動多出450這一格。功率可能分布在附近格子，最高格接近450卻不必恰等於450。我們找到440是因為合成頻率剛好落第11格，方便核對。增加n_fft通常讓格距更小，若只是補零而不增加真實觀察長度，則不能憑空分辨原框沒有足夠資料區分的相近頻率。讀頻譜時先知道分析尺度，才不會把格號換算的小數當成無限精確的頻率測量。

練習只把440改880，先預測峰從第11格到22格、形狀仍(201,21)，再執行核對。頻率變了，但錄音時長、取樣率與框設置沒變，因此兩條軸的長度保持相同。
